In [2]:
# Import google Drive
#from google.colab import drive
#drive.mount('/drive')

In [3]:
%%bash
#rm -rf /content/aggregated/images/train
#rm -rf /content/aggregated/labels/train
mkdir -p /content/aggregated/images /content/aggregated/labels
mkdir -p /content/aggregated/images/train /content/aggregated/images/val /content/aggregated/images/test
mkdir -p /content/aggregated/labels/train /content/aggregated/labels/val /content/aggregated/labels/test

In [ ]:
%%bash

# set the NREC set name
export NREC_IMG_SET_1=apples_left_labeled_1
export NREC_IMG_SET_2=apples_left_labeled_2
export NREC_IMG_SET_3=apples_left_labeled_3
mkdir apples_left_labeled
cp /drive/MyDrive/AgriImages/NREC/$NREC_IMG_SET_1.zip /content/apples_left_labeled
cp /drive/MyDrive/AgriImages/NREC/$NREC_IMG_SET_2.zip /content/apples_left_labeled
cp /drive/MyDrive/AgriImages/NREC/$NREC_IMG_SET_3.zip /content/apples_left_labeled
cd apples_left_labeled
unzip $NREC_IMG_SET_1.zip
unzip $NREC_IMG_SET_2.zip
unzip $NREC_IMG_SET_3.zip

In [5]:
import os
import shutil
from pathlib import Path

def nrec_select_images(src_base_dir:str='/content/apples_left_labeled',
                       dst_base_dir:str='/content/aggregated',
                       type_dir:str='train',
                       percentage_to_copy:float=1.0,
                       positive:bool=True):


    if(positive):
        src_base_dir_full=src_base_dir+"/"+type_dir+"/positive/2015.11Soergels"
    else: 
        src_base_dir_full=src_base_dir+"/"+type_dir+"/negative/2015.11Soergels"

    dirs = [ f.name for f in os.scandir(src_base_dir_full) if f.is_dir()]


    total_img_copied=0

    for d in dirs:
        cur_scen_dir = src_base_dir_full+ "/" +d
        print(" INFO - Current Scenario: "+cur_scen_dir)

        (root,dirs,img_files) =next(os.walk(cur_scen_dir+"/Images"))
        (root,dirs,annotation_files) =next(os.walk(cur_scen_dir+"/Annotations"))

        total_images = len(img_files)
        total_annotations = len(annotation_files)
        img_to_copy_cnt = int(total_images*percentage_to_copy)
        pace = total_images//img_to_copy_cnt



        if total_annotations == total_images :
            print (f'INFO - total_images [{total_images}] same as total_files [{total_images}] only {img_to_copy_cnt} will be copied. pace {pace}')
        else:
            print (f'WARN - total_images {total_images} not same as total_files {total_annotations}')

        
        count=0
        for file_name in sorted(img_files):
            if count%pace==0:

                # remove the prefix "".png"
                cmn_file_name =file_name[:-4]

                img_full_src_path = cur_scen_dir+'/Images/'+cmn_file_name+".png"
                img_full_dst_path = dst_base_dir+'/images/'+type_dir+"/"+cmn_file_name+".png"


                annotation_full_src_path = cur_scen_dir+'/Annotations/'+cmn_file_name+".xml"
                annotation_full_dst_path = dst_base_dir+'/labels/'+type_dir+"/"+cmn_file_name+".xml"

                if(Path(img_full_src_path).exists() & Path(annotation_full_src_path).exists()) :
                    shutil.copy(img_full_src_path,img_full_dst_path)
                    if positive==True:
                        shutil.copy(annotation_full_src_path,annotation_full_dst_path)

                    total_img_copied=total_img_copied+1
                else:
                    print(f"Error - missing files {img_full_src_path} or {annotation_full_src_path}")

            count=count+1

        print(f'Total images copied [{total_img_copied}]')





In [ ]:
# train positive
nrec_select_images(src_base_dir='/content/apples_left_labeled',
                   dst_base_dir='/content/aggregated',
                   type_dir='train',
                   percentage_to_copy=1.0,
                   positive=True)

In [ ]:
# val positive
nrec_select_images(src_base_dir='/content/apples_left_labeled',
                   dst_base_dir='/content/aggregated',
                   type_dir='test',
                   percentage_to_copy=1.0,
                   positive=True)

In [ ]:
# test positive
nrec_select_images(src_base_dir='/content/apples_left_labeled',
                   dst_base_dir='/content/aggregated',
                   type_dir='val',
                   percentage_to_copy=1.0,
                   positive=True)